# Currency Converter Agent AI

In [18]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.messages import HumanMessage
from langchain.tools import tool,InjectedToolArg
from typing import Annotated
import requests
import os
import json


In [3]:
load_dotenv()
model=ChatOpenAI()



In [4]:
#Loading api from env
api_key=os.getenv("CURRENCY_API_KEY")

# Check if it loaded properly
if not api_key:
    print("❌ API key not found. Check your .env file.")
else:
    print("✅ API key loaded successfully!")
    print(f"Key starts with: {api_key[:5]}...")

✅ API key loaded successfully!
Key starts with: 717b2...


In [5]:
@tool 
def get_currency_factor(base_currency:str,target_currency:str,api_key:str =api_key)->float:
    """
    This function fetches the currency conversion factor between a given base factor and target currency
    """
    url=f"https://api.exchangeratesapi.io/v1/latest?access_key={api_key}&format=1"
    response=requests.get(url)
    return response.json()
    

In [6]:
get_currency_factor.invoke({'base_currency':'USD','target_currency':'NPR'})

{'success': True,
 'timestamp': 1773658568,
 'base': 'EUR',
 'date': '2026-03-16',
 'rates': {'AED': 4.212812,
  'AFN': 72.268228,
  'ALL': 96.18316,
  'AMD': 433.226873,
  'ANG': 2.053448,
  'AOA': 1051.913296,
  'ARS': 1603.931883,
  'AUD': 1.627693,
  'AWG': 2.064824,
  'AZN': 1.955783,
  'BAM': 1.959531,
  'BBD': 2.314307,
  'BDT': 140.998501,
  'BGN': 1.96079,
  'BHD': 0.43319,
  'BIF': 3411.195871,
  'BMD': 1.147124,
  'BND': 1.4704,
  'BOB': 7.94012,
  'BRL': 6.176524,
  'BSD': 1.149088,
  'BTC': 1.5560602e-05,
  'BTN': 106.064439,
  'BWP': 15.657885,
  'BYN': 3.399975,
  'BYR': 22483.639138,
  'BZD': 2.310901,
  'CAD': 1.570871,
  'CDF': 2589.059676,
  'CHF': 0.904267,
  'CLF': 0.026748,
  'CLP': 1056.145413,
  'CNY': 7.911252,
  'CNH': 7.908437,
  'COP': 4238.64777,
  'CRC': 540.631871,
  'CUC': 1.147124,
  'CUP': 30.398798,
  'CVE': 110.475858,
  'CZK': 24.409203,
  'DJF': 204.619653,
  'DKK': 7.471698,
  'DOP': 70.59474,
  'DZD': 151.89503,
  'EGP': 60.109155,
  'ERN': 17.20

In [7]:
#Creating conversion function
@tool
def conversion_rate(base_currency:str,target_currency:str):
    """
    This tool only fetches exchange rates.
    Use the currency_convert tool afterwards to calculate conversion.
    
    """
    
    api_key=os.getenv("CURRENCY_API_KEY")
    if not api_key:
        return {
            "success": False,
            "error": "API key not configured"
        }
    
    # Make API call
    url = f"https://api.exchangeratesapi.io/v1/latest?access_key={api_key}&format=1"
    
    try:
        response=requests.get(url)
        data = response.json()
        if data.get('success'):
            rates=data.get('rates',{})
            
            #Convert currency string into upper case
            base_currency=base_currency.upper()
            target_currency=target_currency.upper()
            
            #Check wheather the currency rate for base and target currency exist  on the list
            if base_currency not in rates:
                return{
                    "success": False,
                        "error": f"Currency '{base_currency}' not found",
                        "available_currencies": list(rates.keys())[:10] 
                }
                
            if target_currency not in rates:
                return{
                    "success": False,
                        "error": f"Currency '{target_currency}' not found",
                        "available_currencies": list(rates.keys())[:10] 
                }
            # Calculate conversion
            base_currency_rate = rates[base_currency]
            target_currency_rate = rates[target_currency]
                
            return base_currency_rate ,target_currency_rate
                
        else:
            return {
                    "success": False,
                    "error": data.get('error', {}).get('message', 'Unknown error')
                }
                                                                                       
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }
@tool
def currency_convert(base_currency_rate:int,target_currency_rate:Annotated[float,InjectedToolArg])->float:
    """    Convert currency using provided exchange rates.

    """
    amount=1.0
    #Conversion rate calculation
    base_rate=base_currency_rate
    target_rate=target_currency_rate
    
    # conversion rate formula
    conversion_rate=target_rate/base_rate
    converted_amount=amount*conversion_rate
        
    return converted_amount
    

In [8]:
#tool_binding
llm_with_tools=model.bind_tools([conversion_rate,currency_convert])

In [13]:
messages=[HumanMessage("what is the conversion factor for the USD and NPR, and based on that can you convert 10 USD into NPR  ")]

In [14]:
AI_message=llm_with_tools.invoke(messages)
AI_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 111, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DK0A8nUTMhK4JfH5pDQcRUrp6OZl2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf659-09f6-72c0-aa02-a376e1079bd9-0', tool_calls=[{'name': 'conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_zJEIe2CFDput3zDvBjzWNlYN', 'type': 'tool_call'}, {'name': 'currency_convert', 'args': {'base_currency_rate': 10}, 'id': 'call_fZrRs1rKWD6raI7Nyz5BQV0k', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens

In [15]:
AI_message.tool_calls

[{'name': 'conversion_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_zJEIe2CFDput3zDvBjzWNlYN',
  'type': 'tool_call'},
 {'name': 'currency_convert',
  'args': {'base_currency_rate': 10},
  'id': 'call_fZrRs1rKWD6raI7Nyz5BQV0k',
  'type': 'tool_call'}]

In [ ]:
for tool_call in AI_message.tool_calls:
    #Execute the first tool call to get the conversion rates
    if tool_call['name']=='conversion_rate':
        tool_message1=conversion_rate.invoke(tool_call)
        #fetch the conversion rate
        conversion_rate=json.loads(tool_message1.content)['target_currency_rate']
        

content='[1.147124, 169.702901]' name='conversion_rate' tool_call_id='call_zJEIe2CFDput3zDvBjzWNlYN'
